# 26 · OpenAI / Anthropic 风格 Tool Use

> **学习目标**：从「文本格式 ReAct」升级到「结构化 Tool Use」—— 让 LLM 输出 **JSON 工具调用**，由 SDK 解析，工程上更稳。理解 OpenAI 与 Anthropic 两家 schema 的差别。
>
> **预备**：25 号 notebook（懂 ReAct）。
>
> **为什么重要**：所有生产 Agent 都用 Tool Use 而非 ReAct 文本格式（解析鲁棒 + 模型原生支持）。但**心智模型仍是 ReAct 那 5 步**，只是格式换了。

**两家 schema 对比**：

| 维度 | OpenAI | Anthropic |
|------|--------|-----------|
| 字段名 | `tools` (list of `{type:'function', function:{...}}`) | `tools` (list of `{name, description, input_schema}`) |
| Tool 输入 schema | JSON Schema | JSON Schema |
| 返回 | `tool_calls` (list of `{id, type:'function', function:{name, arguments}}`) | `content` 数组里含 `tool_use` block |
| Tool 结果 | `role='tool'` message | `role='user'` message 里含 `tool_result` block |

**本机走 Ollama 的 OpenAI 兼容接口**（已实测：Ollama `/v1/chat/completions` 支持 tools 字段）。

In [1]:
MODE = 'OFFLINE'        # 'ONLINE' 用本机 Ollama

import json, requests
from datetime import datetime
OLLAMA = 'http://127.0.0.1:11434'

if MODE == 'ONLINE':
    try: requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status(); print('✅ ONLINE')
    except Exception: MODE = 'OFFLINE'; print('⚠ Ollama 未启动，降级 OFFLINE')
if MODE == 'OFFLINE':
    print('OFFLINE：用规则模拟 LLM 的 JSON 工具调用')

OFFLINE：用规则模拟 LLM 的 JSON 工具调用


D:\ProgramData\anaconda3\envs\rag\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


## 1. 工具定义 —— OpenAI 风格 `tools` schema

**关键**：用 **JSON Schema** 声明 input 类型。模型按 schema 生成 arguments，**类型安全**。

In [2]:
# OpenAI 风格 tools 定义
TOOLS_OPENAI = [
    {
        'type': 'function',
        'function': {
            'name': 'calculate',
            'description': '计算一个 Python 数学表达式',
            'parameters': {
                'type': 'object',
                'properties': {
                    'expression': {
                        'type': 'string',
                        'description': '要计算的表达式，如 "2+2" 或 "100*1.05"',
                    },
                },
                'required': ['expression'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': '获取指定城市的当前天气',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {'type': 'string', 'description': '城市名称（中文或英文）'},
                    'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'default': 'celsius'},
                },
                'required': ['city'],
            },
        },
    },
]

# Anthropic 风格 —— 字段名不同但本质等价
TOOLS_ANTHROPIC = [
    {
        'name': t['function']['name'],
        'description': t['function']['description'],
        'input_schema': t['function']['parameters'],
    }
    for t in TOOLS_OPENAI
]

print('OpenAI 第 1 个 tool:')
print(json.dumps(TOOLS_OPENAI[0], ensure_ascii=False, indent=2))
print('\nAnthropic 第 1 个 tool:')
print(json.dumps(TOOLS_ANTHROPIC[0], ensure_ascii=False, indent=2))

OpenAI 第 1 个 tool:
{
  "type": "function",
  "function": {
    "name": "calculate",
    "description": "计算一个 Python 数学表达式",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "要计算的表达式，如 \"2+2\" 或 \"100*1.05\""
        }
      },
      "required": [
        "expression"
      ]
    }
  }
}

Anthropic 第 1 个 tool:
{
  "name": "calculate",
  "description": "计算一个 Python 数学表达式",
  "input_schema": {
    "type": "object",
    "properties": {
      "expression": {
        "type": "string",
        "description": "要计算的表达式，如 \"2+2\" 或 \"100*1.05\""
      }
    },
    "required": [
      "expression"
    ]
  }
}


In [3]:
# 工具实现 —— 真业务函数
import re

def tool_calculate(expression: str) -> str:
    if not re.fullmatch(r'[\d\s+\-*/().%]+', expression):
        return f'ERROR: 表达式含不允许字符'
    try: return str(eval(expression))
    except Exception as e: return f'ERROR: {type(e).__name__}'

def tool_get_weather(city: str, unit: str = 'celsius') -> str:
    # mock 数据；生产里换真 API
    weather_db = {
        '北京': {'temp_c': 18, 'desc': '多云'},
        '上海': {'temp_c': 22, 'desc': '小雨'},
        '广州': {'temp_c': 28, 'desc': '晴'},
        'New York': {'temp_c': 12, 'desc': 'cloudy'},
    }
    if city not in weather_db:
        return json.dumps({'error': f'unknown city: {city}'}, ensure_ascii=False)
    w = weather_db[city]
    temp = w['temp_c'] if unit == 'celsius' else round(w['temp_c'] * 9/5 + 32, 1)
    return json.dumps({'city': city, 'temp': temp, 'unit': unit, 'desc': w['desc']}, ensure_ascii=False)

TOOL_FNS = {'calculate': tool_calculate, 'get_weather': tool_get_weather}

# Smoke
print(tool_calculate('100 * 1.05'))
print(tool_get_weather('北京'))
print(tool_get_weather('北京', 'fahrenheit'))
print(tool_get_weather('火星'))

105.0
{"city": "北京", "temp": 18, "unit": "celsius", "desc": "多云"}
{"city": "北京", "temp": 64.4, "unit": "fahrenheit", "desc": "多云"}
{"error": "unknown city: 火星"}


## 2. LLM with tools —— OFFLINE / ONLINE

**OFFLINE 模拟**：返回 OpenAI 兼容的 response 结构（含 `tool_calls`），让 Agent loop 用同一份解析代码。
**ONLINE**：调 Ollama `/v1/chat/completions` 传 `tools`。

In [4]:
import uuid

def llm_offline(messages: list[dict], tools: list[dict]) -> dict:
    """模拟 LLM：基于规则决定调哪个工具，返回 OpenAI 兼容结构。"""
    user_q = next((m['content'] for m in reversed(messages) if m['role'] == 'user' and isinstance(m['content'], str)), '')
    # 已经调用过的工具
    called = [m for m in messages if m['role'] == 'tool']

    # 规则路由
    if called:
        # 已有工具结果 → 综合给答案
        last_tool_result = called[-1]['content']
        return {'role': 'assistant', 'content': f'根据工具结果 "{last_tool_result}"，答案如下。', 'tool_calls': None}

    weather_m = re.search(r'(北京|上海|广州|New York|纽约|火星)\s*(?:的)?(?:天气|气温|温度)', user_q)
    calc_m = re.search(r'([\d.]+\s*[+\-*/]\s*[\d.]+(?:\s*[+\-*/]\s*[\d.]+)*)', user_q)

    if weather_m:
        city = weather_m.group(1).replace('纽约', 'New York')
        unit = 'fahrenheit' if '华氏' in user_q else 'celsius'
        return {'role': 'assistant', 'content': '', 'tool_calls': [{
            'id': f'call_{uuid.uuid4().hex[:8]}', 'type': 'function',
            'function': {'name': 'get_weather', 'arguments': json.dumps({'city': city, 'unit': unit}, ensure_ascii=False)},
        }]}
    if calc_m:
        return {'role': 'assistant', 'content': '', 'tool_calls': [{
            'id': f'call_{uuid.uuid4().hex[:8]}', 'type': 'function',
            'function': {'name': 'calculate', 'arguments': json.dumps({'expression': calc_m.group(1).strip()})},
        }]}
    return {'role': 'assistant', 'content': f'(OFFLINE) 不知道怎么回答 "{user_q}"', 'tool_calls': None}

def llm_online(messages, tools):
    r = requests.post(f'{OLLAMA}/v1/chat/completions', json={
        'model': 'qwen1.5_1.8',
        'messages': messages,
        'tools': tools,
        'tool_choice': 'auto',
        'temperature': 0,
    }, timeout=120)
    r.raise_for_status()
    return r.json()['choices'][0]['message']

llm = llm_online if MODE == 'ONLINE' else llm_offline

## 3. Agent loop —— 结构化 tool_calls 版本

与 25 号文本格式版本对比，**核心循环结构没变**，只是「解析 Action」换成「读 message.tool_calls 字段」。

In [5]:
def run_agent(question: str, tools: list[dict], tool_fns: dict, max_iter: int = 6, verbose: bool = True) -> dict:
    messages = [
        {'role': 'system', 'content': '你是助手，使用工具回答用户。不知道就说不知道，不要编造。'},
        {'role': 'user',   'content': question},
    ]
    trace = []
    for step in range(1, max_iter + 1):
        msg = llm(messages, tools)
        trace.append({'step': step, 'message': msg})
        if verbose:
            print(f'\n── step {step} ──')
            print(f'LLM> content={msg.get("content", "")[:80]!r}  tool_calls={msg.get("tool_calls")}')

        tool_calls = msg.get('tool_calls')
        if not tool_calls:
            return {'answer': msg.get('content', ''), 'trace': trace, 'iter': step}

        # 必须把 assistant 整条 message append（包括 tool_calls）
        messages.append(msg)

        # 逐个执行 tool_call，把结果以 role='tool' 加回
        for tc in tool_calls:
            name = tc['function']['name']
            args = json.loads(tc['function']['arguments']) if isinstance(tc['function']['arguments'], str) else tc['function']['arguments']
            if name in tool_fns:
                result = tool_fns[name](**args)
            else:
                result = f'ERROR: unknown tool {name!r}'
            if verbose:
                print(f'TOOL[{name}]({args})> {result}')
            messages.append({
                'role': 'tool',
                'tool_call_id': tc['id'],
                'content': result,
            })
            trace[-1].setdefault('tool_results', []).append({'name': name, 'args': args, 'result': result})
    return {'answer': f'(达到 max_iter={max_iter})', 'trace': trace, 'iter': max_iter}

In [6]:
for q in ['北京现在的天气怎么样', '纽约的华氏温度是多少', '帮我算 250 * 1.05', '火星气温']:
    print(f'\n{"="*50}\n📝 Q: {q}')
    result = run_agent(q, TOOLS_OPENAI, TOOL_FNS, max_iter=3, verbose=True)
    print(f'\n✅ 答案: {result["answer"]}')


📝 Q: 北京现在的天气怎么样

── step 1 ──
LLM> content='(OFFLINE) 不知道怎么回答 "北京现在的天气怎么样"'  tool_calls=None

✅ 答案: (OFFLINE) 不知道怎么回答 "北京现在的天气怎么样"

📝 Q: 纽约的华氏温度是多少

── step 1 ──
LLM> content='(OFFLINE) 不知道怎么回答 "纽约的华氏温度是多少"'  tool_calls=None

✅ 答案: (OFFLINE) 不知道怎么回答 "纽约的华氏温度是多少"

📝 Q: 帮我算 250 * 1.05

── step 1 ──
LLM> content=''  tool_calls=[{'id': 'call_1362b171', 'type': 'function', 'function': {'name': 'calculate', 'arguments': '{"expression": "250 * 1.05"}'}}]
TOOL[calculate]({'expression': '250 * 1.05'})> 262.5

── step 2 ──
LLM> content='根据工具结果 "262.5"，答案如下。'  tool_calls=None

✅ 答案: 根据工具结果 "262.5"，答案如下。

📝 Q: 火星气温

── step 1 ──
LLM> content=''  tool_calls=[{'id': 'call_6bcd68b0', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city": "火星", "unit": "celsius"}'}}]
TOOL[get_weather]({'city': '火星', 'unit': 'celsius'})> {"error": "unknown city: 火星"}

── step 2 ──
LLM> content='根据工具结果 "{"error": "unknown city: 火星"}"，答案如下。'  tool_calls=None

✅ 答案: 根据工具结果 "{"error": "unknown city

## 4. parallel_tool_calls —— 一次调多个工具

**真实场景**：用户问「北京和上海今天哪个更暖」→ Agent 应**并行**调两次 `get_weather`，而不是串行。

OpenAI / Anthropic 都默认支持 `parallel_tool_calls=True`，response 的 `tool_calls` 会是**列表**。我们的 Agent 已经按列表处理（for tc in tool_calls），所以**自动支持并行**。

In [7]:
# OFFLINE 模拟并行：手动构造一个含 2 个 tool_calls 的 LLM 输出
def llm_offline_parallel(messages, tools):
    user_q = next((m['content'] for m in reversed(messages) if m['role'] == 'user' and isinstance(m['content'], str)), '')
    cities = re.findall(r'(北京|上海|广州)', user_q)
    called = [m for m in messages if m['role'] == 'tool']
    if called:
        # 已有结果：综合
        results = [m['content'] for m in called]
        return {'role': 'assistant', 'content': f'结果比较: {"; ".join(results)}', 'tool_calls': None}
    if len(cities) >= 2 and ('比较' in user_q or '哪个' in user_q):
        # 并行调 2 个 get_weather
        return {'role': 'assistant', 'content': '', 'tool_calls': [
            {'id': f'c{i}', 'type': 'function',
             'function': {'name': 'get_weather', 'arguments': json.dumps({'city': c}, ensure_ascii=False)}}
            for i, c in enumerate(cities)
        ]}
    return llm_offline(messages, tools)

if MODE == 'OFFLINE':
    saved, llm = llm, llm_offline_parallel
    result = run_agent('北京和上海今天哪个更暖', TOOLS_OPENAI, TOOL_FNS, max_iter=3, verbose=True)
    llm = saved
    print(f'\n✅ 并行 tool calls: {len(result["trace"][0].get("tool_results", []))} 个工具同时调')
else:
    print('（ONLINE 自动并行，已在上一个 cell 体现）')


── step 1 ──
LLM> content=''  tool_calls=[{'id': 'c0', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city": "北京"}'}}, {'id': 'c1', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city": "上海"}'}}]
TOOL[get_weather]({'city': '北京'})> {"city": "北京", "temp": 18, "unit": "celsius", "desc": "多云"}
TOOL[get_weather]({'city': '上海'})> {"city": "上海", "temp": 22, "unit": "celsius", "desc": "小雨"}

── step 2 ──
LLM> content='结果比较: {"city": "北京", "temp": 18, "unit": "celsius", "desc": "多云"}; {"city": "上海"'  tool_calls=None

✅ 并行 tool calls: 2 个工具同时调


## 5. 与 ReAct 文本格式对比

| 维度 | ReAct 文本（25 号） | Tool Use 结构化（本号） |
|------|-------------------|----------------------|
| 解析 | regex / 字符串 | JSON，**类型安全** |
| 模型支持 | 任何 LLM（仅靠 prompt） | 需模型支持 function-calling |
| 错误处理 | 解析失败要兜底 | 客户端 SDK 已处理大部分 |
| 并行调用 | 困难 | 原生支持 `parallel_tool_calls` |
| 流式输出 | 困难 | 原生支持 tool_calls 流式增量 |
| 学习价值 | **高**：理解 Agent 本质 | 高：实战必备 |

**结论**：教学先学 ReAct（理解原理），工程上用 Tool Use（更稳）。**心智模型是同一个**：LLM + tools + loop。

## 深入思考

1. **OpenAI / Anthropic schema 看上去差不多，迁移有什么坑？**
   - 字段名 (`function` vs `input_schema`)、tool_result 回填方式 (`role='tool'` vs `tool_result` block)、id 字段命名都不同。但**业务逻辑（工具函数本身）零修改**，只换 client wrapper 那一层。
2. **`tool_choice` 几种取值？**
   - `'auto'`（LLM 自己决定）/ `'none'`（强制不用工具）/ `'required'`（必须用工具）/ `{'function':{'name':'xxx'}}`（强制指定工具）。**评估时常用 `required` 测「该用工具时是否真用」**。
3. **`tool_calls` 里 LLM 给的 `arguments` 是不是一定符合 schema？**
   - 不一定。LLM 可能漏字段 / 类型错。**生产里必须 try/except + JSON Schema 验证**。`pydantic.parse_obj_as` 是常用兜底。
4. **并行 tool_calls 真的并行执行了吗？**
   - 由你决定。LLM 给的是「可以并行」的多个 calls，你 for 循环串行执行也行，用 `asyncio.gather` 真并行也行 —— **后者是生产标配**。
5. **Anthropic 的 prompt caching 跟 tool_use 怎么搭？**
   - 把 `tools` 数组放进 system + prompt cache，**省钱大头**。每次 query 只 cache miss 一点点（用户消息）。

**改一改**：
- 把 `tool_choice` 改成 `'required'`，跑「你好」这种用不上工具的 query，看 LLM 是否被迫硬调（OFFLINE 不会，ONLINE 真 LLM 会）
- 把 `get_weather` 的 `unit` 参数加个 `default='celsius'`，故意不传 unit，看 OFFLINE 是否仍能工作

## 自检 ✅

- [ ] 不查代码写出 OpenAI tools 数组的最小 schema
- [ ] 解释 `role='tool'` message 在对话历史里的位置和必要性
- [ ] 解释 OpenAI / Anthropic 两家差别的 3 个关键点
- [ ] 解释 `parallel_tool_calls` 何时省时间、何时反而麻烦
- [ ] 给一个 Tool Use 失败的 case（如 arguments 不符 schema），列 3 个排查方向

## 下一步

→ [`27_observe_claude_code.ipynb`](27_observe_claude_code.ipynb)